# GDAL and Raster Utilities

This notebook contains various utility functions for working with raster data, particularly TIFF and COG (Cloud Optimized GeoTIFF) files.

**Utilities covered:**
- 🔍 **TIFF file inspection** and metadata extraction
- 🌐 **GDAL capability checking** (WebP support, version info)
- 📦 **File decompression** and format conversion
- 🛠️ **Batch processing** for multiple files
- 📊 **Format analysis** and optimization checks

## Prerequisites

Install required dependencies:
```bash
pip install gdal rasterio
```

## 1. Import Dependencies and Check GDAL Installation

In [ ]:
import os
import sys
from pathlib import Path
from typing import List, Dict, Any

# GDAL imports
try:
    from osgeo import gdal
    GDAL_AVAILABLE = True
    print(f"✅ GDAL {gdal.VersionInfo('RELEASE_NAME')} available")
except ImportError:
    GDAL_AVAILABLE = False
    print("❌ GDAL not available")

# Rasterio imports
try:
    import rasterio
    from rasterio.env import GDALVersion
    RASTERIO_AVAILABLE = True
    print(f"✅ Rasterio {rasterio.__version__} available")
    print(f"   Using GDAL {rasterio.__gdal_version__}")
except ImportError:
    RASTERIO_AVAILABLE = False
    print("❌ Rasterio not available")

if not (GDAL_AVAILABLE or RASTERIO_AVAILABLE):
    print("⚠️  No raster processing libraries available!")
    print("   Install with: pip install gdal rasterio")

## 2. GDAL Capability Checker

In [ ]:
def check_gdal_capabilities():
    """Check GDAL installation and supported formats."""
    
    if not GDAL_AVAILABLE:
        print("❌ GDAL not available for capability checking")
        return
    
    print("🔍 === GDAL Capabilities Check ===")
    
    # Version information
    print(f"\n📦 Version Information:")
    print(f"   GDAL Version: {gdal.VersionInfo('RELEASE_NAME')}")
    print(f"   Version Number: {gdal.VersionInfo('VERSION_NUM')}")
    print(f"   Build Info: {gdal.VersionInfo('BUILD_INFO')}")
    
    # Check for WebP support (commonly needed)
    print(f"\n🌐 Format Support:")
    formats = gdal.GetDriverCount()
    
    # List of important formats to check
    important_formats = {
        'WEBP': 'WebP image format',
        'JPEG': 'JPEG image format',
        'PNG': 'PNG image format',
        'GTiff': 'GeoTIFF format',
        'COG': 'Cloud Optimized GeoTIFF',
        'NetCDF': 'Network Common Data Format',
        'HDF5': 'Hierarchical Data Format 5'
    }
    
    supported_formats = {}
    for i in range(formats):
        driver = gdal.GetDriver(i)
        driver_name = driver.GetDescription()
        supported_formats[driver_name] = True
    
    for format_name, description in important_formats.items():
        status = "✅" if format_name in supported_formats else "❌"
        print(f"   {status} {format_name}: {description}")
    
    # WebP specific check
    webp_supported = 'WEBP' in supported_formats
    if webp_supported:
        print(f"\n🎉 GDAL has WebP support!")
    else:
        print(f"\n⚠️  GDAL does NOT have WebP support.")
        print(f"   You may need to install GDAL with WebP support.")
    
    # List all available drivers (first 20)
    print(f"\n📋 Available Drivers (showing first 20 of {formats}):")
    for i in range(min(20, formats)):
        driver = gdal.GetDriver(i)
        print(f"   {i+1:2d}. {driver.GetDescription()}")
    
    if formats > 20:
        print(f"   ... and {formats - 20} more drivers")

# Run the capability check
check_gdal_capabilities()

## 3. Rasterio Environment Check

In [ ]:
def check_rasterio_environment():
    """Check rasterio environment and driver support."""
    
    if not RASTERIO_AVAILABLE:
        print("❌ Rasterio not available for environment checking")
        return
    
    print("\n🔍 === Rasterio Environment Check ===")
    
    # Version information
    print(f"\n📦 Version Information:")
    print(f"   Rasterio Version: {rasterio.__version__}")
    print(f"   GDAL Version: {rasterio.__gdal_version__}")
    
    runtime_version = GDALVersion.runtime()
    print(f"   Runtime GDAL: {runtime_version.major}.{runtime_version.minor}")
    
    # Check environment and drivers
    with rasterio.Env() as env:
        print(f"\n🌐 Environment Details:")
        print(f"   GDAL Data: {env.get('GDAL_DATA', 'Not set')}")
        print(f"   Proj Data: {env.get('PROJ_DATA', 'Not set')}")
        
        # Available drivers
        available_drivers = env.drivers()
        print(f"\n📋 Available Drivers: {len(available_drivers.keys())} total")
        
        # Check for specific driver support
        important_drivers = ['WEBP', 'JPEG', 'PNG', 'GTiff', 'COG', 'NetCDF']
        print(f"\n🎯 Important Driver Support:")
        for driver in important_drivers:
            status = "✅" if driver in available_drivers else "❌"
            print(f"   {status} {driver}")
        
        # Show sample of available drivers
        print(f"\n📄 Sample of Available Drivers:")
        driver_list = list(available_drivers.keys())
        for i, driver in enumerate(driver_list[:15]):
            mode = available_drivers[driver]
            print(f"   {i+1:2d}. {driver} ({mode})")
            if len(driver_list) > 15:
            print(f"   ... and {len(driver_list) - 15} more")

# Run the rasterio environment check
check_rasterio_environment()

## 4. TIFF File Inspector

In [ ]:
def describe_tiff(file_path: str) -> Dict[str, Any]:
    """Describe TIFF file properties using GDAL."""
    
    if not GDAL_AVAILABLE:
        print("❌ GDAL not available for TIFF inspection")
        return {}
    
    tiff_path = Path(file_path)
    if not tiff_path.exists():
        print(f"❌ File not found: {tiff_path}")
        return {}
        
    print(f"🔍 === Analyzing TIFF File: {tiff_path.name} ===")
    
    ds = gdal.Open(str(tiff_path), gdal.GA_ReadOnly)
    if ds is None:
        print(f"❌ Unable to open dataset: {tiff_path}")
        return {}

    try:
        driver = ds.GetDriver()
        geotransform = ds.GetGeoTransform(can_return_null=True)

        # Basic information
        basic_info = {
            'path': str(tiff_path),
            'driver': driver.ShortName if driver else 'unknown',
            'width': ds.RasterXSize,
            'height': ds.RasterYSize,
            'band_count': ds.RasterCount
        }
        
        print(f"\n📊 === Basic Information ===")
        print(f"   Path: {basic_info['path']}")
        print(f"   Driver: {basic_info['driver']}")
        print(f"   Dimensions: {basic_info['width']} x {basic_info['height']} pixels")
        print(f"   Bands: {basic_info['band_count']}")
        
        # File size
        file_size_mb = tiff_path.stat().st_size / (1024 * 1024)
        print(f"   File size: {file_size_mb:.2f} MB")

        # Band details
        band_info = []
        overview_info = []
        
        print(f"\n🎵 === Band Information ===")
        for band_index in range(1, ds.RasterCount + 1):
            band = ds.GetRasterBand(band_index)
            if band is None:
                continue

            band_data = {
                'index': band_index,
                'dtype': gdal.GetDataTypeName(band.DataType),
                'nodata': band.GetNoDataValue(),
                'color_interp': gdal.GetColorInterpretationName(band.GetColorInterpretation()),
                'block_size': band.GetBlockSize()
            }
            band_info.append(band_data)
            
            print(f"   Band {band_index}:")
            print(f"      Data type: {band_data['dtype']}")
            print(f"      NoData: {band_data['nodata']}")
            print(f"      Color interpretation: {band_data['color_interp']}")
            print(f"      Block size: {band_data['block_size']}")

            # Overview information
            overview_count = band.GetOverviewCount()
            if overview_count > 0:
                factors = []
                for i in range(overview_count):
                    overview = band.GetOverview(i)
                    if overview is None or overview.XSize == 0:
                        continue
                    factors.append(max(1, round(ds.RasterXSize / overview.XSize)))
                print(f"      Overviews: {factors if factors else 'none'}")
                overview_info.append({'band': band_index, 'factors': factors})
            else:
                print(f"      Overviews: none")

        # Spatial information
        spatial_info = {}
        print(f"\n🌍 === Spatial Information ===")
        
        projection = ds.GetProjectionRef() or "None"
        spatial_info['crs'] = projection
        print(f"   CRS: {projection if projection != 'None' else 'No CRS defined'}")

        if geotransform:
            x_min = geotransform[0]
            y_max = geotransform[3]
            x_res = geotransform[1]
            y_res = geotransform[5]
            x_max = x_min + ds.RasterXSize * x_res
            y_min = y_max + ds.RasterYSize * y_res
            
            spatial_info.update({
                'bounds': (x_min, y_min, x_max, y_max),
                'resolution': (x_res, abs(y_res)),
                'transform': geotransform
            })
            
            print(f"   Bounds: ({x_min:.2f}, {y_min:.2f}, {x_max:.2f}, {y_max:.2f})")
            print(f"   Resolution: ({x_res:.6f}, {abs(y_res):.6f})")
            print(f"   Pixel size: {abs(x_res * y_res):.2e} square units")
        else:
            print(f"   Bounds: No geotransform available")
            print(f"   Resolution: No geotransform available")

        # TIFF structure information
        print(f"\n🏗️  === TIFF Structure ===")
        image_structure = ds.GetMetadata("IMAGE_STRUCTURE") or {}
        
        structure_info = {
            'tiled': image_structure.get('TILED', 'NO'),
            'compression': image_structure.get('COMPRESSION', 'unknown'),
            'interleaving': image_structure.get('INTERLEAVE', 'unknown')
        }
        
        print(f"   Tiled: {structure_info['tiled']}")
        print(f"   Compression: {structure_info['compression']}")
        print(f"   Interleaving: {structure_info['interleaving']}")
        
        # Check if it's a Cloud Optimized GeoTIFF
        is_cog = (
            structure_info['tiled'] == 'YES' and
            len(overview_info) > 0 and
            all(len(ov['factors']) > 0 for ov in overview_info)
        )
        
        cog_status = "✅ Likely COG" if is_cog else "❓ Not optimized as COG"
        print(f"   COG Status: {cog_status}")

        # Additional metadata
        tags = ds.GetMetadata() or {}
        if tags:
            print(f"\n🏷️  === Additional Metadata ===")
            for key in sorted(list(tags.keys())[:10]):  # Show first 10 tags
                value = tags[key]
                if len(value) > 100:
                    value = value[:97] + "..."
                print(f"   {key}: {value}")
            
            if len(tags) > 10:
                print(f"   ... and {len(tags) - 10} more metadata fields")
        
        # Compile complete information
        complete_info = {
            'basic': basic_info,
            'bands': band_info,
            'spatial': spatial_info,
            'structure': structure_info,
            'overviews': overview_info,
            'metadata': tags,
            'is_cog': is_cog,
            'file_size_mb': file_size_mb
        }
        
        return complete_info
        
    finally:
        ds = None  # Close dataset

# Example usage function
def analyze_sample_file():
    """Analyze a sample TIFF file if available."""
    
    # Look for sample TIFF files in common locations
    sample_locations = [
        "sample.tif",
        "sample.tiff",
        "test.tif",
        "linz_sample.tif",
        "rasterio_subset_example.tif"
    ]
    
    sample_file = None
    for location in sample_locations:
        if Path(location).exists():
            sample_file = location
            break
    
    if sample_file:
        print(f"📁 Found sample file: {sample_file}")
        result = describe_tiff(sample_file)
        return result
    else:
        print(f"📁 No sample TIFF files found in current directory")
        print(f"   Looking for: {', '.join(sample_locations)}")
        print(f"   💡 You can create sample files using previous notebooks")
        return None

# Try to analyze a sample file
sample_analysis = analyze_sample_file()

print("\n✅ TIFF inspector ready! Use describe_tiff(file_path) to analyze any TIFF file.")

## 5. TIFF Decompression Utility

In [ ]:
def uncompress_tiff(input_path: str, output_path: str, 
                   tile_size: int = 512, bigtiff: str = "IF_SAFER") -> bool:
    """Decompress a TIFF file and save as uncompressed GeoTIFF."""
    
    if not GDAL_AVAILABLE:
        print("❌ GDAL not available for TIFF decompression")
        return False
    
    input_path = Path(input_path)
    output_path = Path(output_path)
    
    if not input_path.exists():
        print(f"❌ Input file not found: {input_path}")
        return False
    
    print(f"🔧 Decompressing: {input_path.name}")
    print(f"   Output: {output_path}")
    
    try:
        # Open input dataset
        ds = gdal.Open(str(input_path), gdal.GA_ReadOnly)
        if ds is None:
            print(f"❌ Could not open: {input_path}")
            return False

        # Get input file info
        input_size_mb = input_path.stat().st_size / (1024 * 1024)
        print(f"   Input size: {input_size_mb:.2f} MB")
        print(f"   Dimensions: {ds.RasterXSize} x {ds.RasterYSize} x {ds.RasterCount}")

        # Create output directory if needed
        output_path.parent.mkdir(parents=True, exist_ok=True)

        # Set up creation options for uncompressed TIFF
        creation_options = [
            "COMPRESS=NONE",
            "TILED=YES",
            f"BLOCKXSIZE={tile_size}",
            f"BLOCKYSIZE={tile_size}",
            f"BIGTIFF={bigtiff}",
        ]
        
        print(f"   Settings:")
        print(f"      Compression: None")
        print(f"      Tiled: Yes ({tile_size}x{tile_size})")
        print(f"      BigTIFF: {bigtiff}")

        # Create output dataset
        driver = gdal.GetDriverByName("GTiff")
        out_ds = driver.CreateCopy(
            str(output_path), ds, 
            strict=0,  # Allow format changes
            options=creation_options
        )
        
        if out_ds is None:
            print(f"❌ Could not write: {output_path}")
            return False

        # Flush and close
        out_ds.FlushCache()
        out_ds = None
        ds = None
        
        # Check output file
        if output_path.exists():
            output_size_mb = output_path.stat().st_size / (1024 * 1024)
            print(f"   ✅ Successfully written: {output_path.name}")
            print(f"   Output size: {output_size_mb:.2f} MB")
            print(f"   Size change: {output_size_mb/input_size_mb:.1f}x {'larger' if output_size_mb > input_size_mb else 'smaller'}")
            return True
        else:
            print(f"❌ Output file was not created")
            return False
            
    except Exception as e:
        print(f"❌ Error during decompression: {e}")
        return False

def batch_uncompress_tiffs(input_folder: str, output_folder: str, 
                          file_pattern: str = "*.tif*") -> Dict[str, bool]:
    """Batch decompress all TIFF files in a folder."""
    
    input_dir = Path(input_folder)
    output_dir = Path(output_folder)
    
    if not input_dir.exists():
        print(f"❌ Input folder not found: {input_dir}")
        return {}
    
    # Create output directory
    output_dir.mkdir(parents=True, exist_ok=True)
    
    # Find TIFF files
    tiff_files = list(input_dir.glob(file_pattern))
    
    if not tiff_files:
        print(f"❌ No TIFF files found in {input_folder} matching {file_pattern}")
        return {}
    
    print(f"📁 === Batch Decompression ===")
    print(f"   Input folder: {input_dir}")
    print(f"   Output folder: {output_dir}")
    print(f"   Files found: {len(tiff_files)}")
    
    results = {}
    successful = 0
    
    for i, tiff_file in enumerate(tiff_files, 1):
        print(f"\n[{i}/{len(tiff_files)}] Processing: {tiff_file.name}")
        
        output_file = output_dir / tiff_file.name
        
        # Skip if already exists
        if output_file.exists():
            print(f"   ⏭️  Already exists, skipping")
            results[tiff_file.name] = True
            successful += 1
            continue
        
        # Process the file
        success = uncompress_tiff(str(tiff_file), str(output_file))
        results[tiff_file.name] = success
        
        if success:
            successful += 1
    
    print(f"\n📊 === Batch Results ===")
    print(f"   Total files: {len(tiff_files)}")
    print(f"   Successful: {successful}")
    print(f"   Failed: {len(tiff_files) - successful}")
    
    if successful < len(tiff_files):
        print(f"\n❌ Failed files:")
        for filename, success in results.items():
            if not success:
                print(f"      {filename}")
    
    return results

# Example usage
print("✅ TIFF decompression utilities ready!")
print("\n💡 Usage examples:")
print("   uncompress_tiff('input.tif', 'output.tif')")
print("   batch_uncompress_tiffs('input_folder', 'output_folder')")

## 6. File Format Analyzer

In [ ]:
def analyze_folder_formats(folder_path: str) -> Dict[str, Any]:
    """Analyze all raster files in a folder and summarize formats."""
    
    folder = Path(folder_path)
    if not folder.exists():
        print(f"❌ Folder not found: {folder}")
        return {}
    
    print(f"📁 === Analyzing Folder: {folder} ===")
    
    # Common raster file extensions
    raster_extensions = {
        '.tif', '.tiff', '.jpg', '.jpeg', '.png', '.jp2', '.webp',
        '.bmp', '.gif', '.hdf', '.nc', '.tif', '.img', '.bil', '.bsq'
    }
    
    # Find all potential raster files
    raster_files = []
    for ext in raster_extensions:
        raster_files.extend(folder.glob(f"*{ext}"))
        raster_files.extend(folder.glob(f"*{ext.upper()}"))
    
    if not raster_files:
        print(f"❌ No raster files found in {folder}")
        return {}
    
    print(f"📊 Found {len(raster_files)} potential raster files")
    
    # Analyze each file
    analysis_results = {
        'total_files': len(raster_files),
        'formats': {},
        'compressions': {},
        'sizes_mb': [],
        'total_size_mb': 0,
        'cog_count': 0,
        'tiled_count': 0,
        'overview_count': 0,
        'files': []
    }
    
    for i, file_path in enumerate(raster_files, 1):
        print(f"\n[{i}/{len(raster_files)}] Analyzing: {file_path.name}")
        
        try:
            file_info = describe_tiff(str(file_path))
            
            if file_info:
                # Extract key information
                basic = file_info.get('basic', {})
                structure = file_info.get('structure', {})
                
                # Track formats
                format_name = basic.get('driver', 'unknown')
                analysis_results['formats'][format_name] = analysis_results['formats'].get(format_name, 0) + 1
                
                # Track compressions
                compression = structure.get('compression', 'unknown')
                analysis_results['compressions'][compression] = analysis_results['compressions'].get(compression, 0) + 1
                
                # Track sizes
                size_mb = file_info.get('file_size_mb', 0)
                analysis_results['sizes_mb'].append(size_mb)
                analysis_results['total_size_mb'] += size_mb
                
                # Track optimization features
                if structure.get('tiled') == 'YES':
                    analysis_results['tiled_count'] += 1
                
                if file_info.get('overviews'):
                    analysis_results['overview_count'] += 1
                
                if file_info.get('is_cog', False):
                    analysis_results['cog_count'] += 1
                
                # Store individual file info
                analysis_results['files'].append({
                    'name': file_path.name,
                    'format': format_name,
                    'compression': compression,
                    'size_mb': size_mb,
                    'is_cog': file_info.get('is_cog', False),
                    'tiled': structure.get('tiled') == 'YES',
                    'has_overviews': bool(file_info.get('overviews'))
                })
                
        except Exception as e:
            print(f"   ❌ Error analyzing {file_path.name}: {e}")
            analysis_results['files'].append({
                'name': file_path.name,
                'error': str(e)
            })
    
    # Print summary
    print(f"\n📈 === Folder Analysis Summary ===")
    
    print(f"\n📊 File Counts:")
    print(f"   Total files: {analysis_results['total_files']}")
    print(f"   Total size: {analysis_results['total_size_mb']:.2f} MB")
    print(f"   Average size: {analysis_results['total_size_mb']/analysis_results['total_files']:.2f} MB")
    
    print(f"\n📄 Formats:")
    for format_name, count in sorted(analysis_results['formats'].items()):
        percentage = count / analysis_results['total_files'] * 100
        print(f"   {format_name}: {count} files ({percentage:.1f}%)")
    
    print(f"\n🗜️  Compression:")
    for compression, count in sorted(analysis_results['compressions'].items()):
        percentage = count / analysis_results['total_files'] * 100
        print(f"   {compression}: {count} files ({percentage:.1f}%)")
    
    print(f"\n⚡ Optimization:")
    cog_percentage = analysis_results['cog_count'] / analysis_results['total_files'] * 100
    tiled_percentage = analysis_results['tiled_count'] / analysis_results['total_files'] * 100
    overview_percentage = analysis_results['overview_count'] / analysis_results['total_files'] * 100
    
    print(f"   COG-optimized: {analysis_results['cog_count']} files ({cog_percentage:.1f}%)")
    print(f"   Tiled: {analysis_results['tiled_count']} files ({tiled_percentage:.1f}%)")
    print(f"   With overviews: {analysis_results['overview_count']} files ({overview_percentage:.1f}%)")
    
    # Recommendations
    print(f"\n💡 Recommendations:")
    if cog_percentage < 50:
        print(f"   📈 Consider converting files to COG format for better performance")
    if tiled_percentage < 80:
        print(f"   📈 Consider tiling files for better random access")
    if overview_percentage < 70:
        print(f"   📈 Consider adding overviews for better performance at multiple scales")
    
    return analysis_results

# Example usage
print("✅ Folder format analyzer ready!")
print("\n💡 Usage example:")
print("   results = analyze_folder_formats('path/to/your/raster/folder')")

# Try to analyze current directory
current_dir_analysis = analyze_folder_formats(".")

## 7. Interactive TIFF Inspector

In [ ]:
def interactive_tiff_inspector():
    """Interactive function to inspect TIFF files."""
    
    print("🔍 === Interactive TIFF Inspector ===")
    print("Enter TIFF file paths to analyze (or 'quit' to exit)")
    
    while True:
        try:
            file_path = input("\n📁 Enter TIFF file path: ").strip()
            
            if file_path.lower() in ['quit', 'exit', 'q']:
                print("👋 Goodbye!")
                break
            
            if not file_path:
                print("⚠️  Please enter a file path")
                continue
            
            # Remove quotes if present
            file_path = file_path.strip('"\'')
            
            # Analyze the file
            result = describe_tiff(file_path)
            
            if result:
                print(f"\n✅ Analysis complete for {Path(file_path).name}")
                
                # Ask if user wants to see detailed info
                show_details = input("\n📋 Show detailed metadata? (y/n): ").strip().lower()
                if show_details in ['y', 'yes']:
                    metadata = result.get('metadata', {})
                    if metadata:
                        print(f"\n🏷️  === All Metadata ===")
                        for key, value in sorted(metadata.items()):
                            if len(str(value)) > 200:
                                value = str(value)[:197] + "..."
                            print(f"   {key}: {value}")
                # Ask if user wants to decompress
                if result.get('structure', {}).get('compression', '').upper() not in ['NONE', 'UNKNOWN']:
                    decompress = input("\n🔧 Decompress this file? (y/n): ").strip().lower()
                    if decompress in ['y', 'yes']:
                        input_path = Path(file_path)
                        output_path = input_path.parent / f"uncompressed_{input_path.name}"
                        
                        success = uncompress_tiff(str(input_path), str(output_path))
                        if success:
                            print(f"✅ Decompressed file saved as: {output_path}")
            
        except KeyboardInterrupt:
            print("\n👋 Interrupted by user")
            break
        except Exception as e:
            print(f"❌ Error: {e}")

# Allow users to start interactive mode
print("✅ Interactive TIFF inspector ready!")
print("\n💡 To start interactive mode, run: interactive_tiff_inspector()")

# Uncomment the next line to start interactive mode immediately
# interactive_tiff_inspector()

## Summary

This notebook provides comprehensive utilities for working with raster data:

### ✅ **Capabilities Covered:**

🔍 **GDAL capability checking** - Verify installation and supported formats  
🌐 **Environment analysis** - Check rasterio and GDAL configuration  
📊 **TIFF file inspection** - Detailed metadata and structure analysis  
🔧 **File decompression** - Convert compressed TIFFs to uncompressed format  
    
,
,
,
,
| `batch_uncompress_tiffs(folder, output)` | Process multiple files |
| `analyze_folder_formats(folder)` | Analyze all rasters in a folder |
| `interactive_tiff_inspector()` | Interactive file inspection |

### 🎯 **Use Cases:**

- **Quality control**: Verify TIFF file integrity and optimization
- **Performance optimization**: Identify files needing COG conversion
- **Format conversion**: Decompress files for specific applications
- **Metadata extraction**: Understand spatial reference and properties
- **Batch processing**: Handle large collections of raster files
- **Environment setup**: Verify GDAL installation and capabilities

### 💡 **Best Practices:**

- Always check GDAL capabilities before processing
- Use COG format for web-based applications
- Include overviews for better performance
- Monitor file sizes during decompression
- Verify spatial reference information
- Keep compressed originals as backups

### 🚀 **Next Steps:**

- Integrate with LINZ data download workflows
- Automate quality control checks
- Build processing pipelines with these utilities
- Combine with visualization tools for data exploration